# quadratic

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/quadratic.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们做一组模拟：设定 600 个观测，让解释变量 x 服从卡方分布（右偏、非负），更贴近收入等实际变量的分布形态。

In [ ]:
%%stata
clear
set obs 600
set seed 505
gen x=rchi2(1)*2
su x

我们设定真实数据生成过程为 y=2-exp(-0.5x)，再加上正态噪声。这是一个递增但逐渐趋平的凸函数（边际效应递减），我们看看用二次函数能否逼近它。

In [ ]:
%%stata
gen y_true=2-exp(-0.5*x)
gen y=y_true+rnormal()*2

我们生成 x 的平方项并做二次回归。二次函数是未知条件期望函数的二阶泰勒近似，可以在一定范围内拟合非线性关系。

In [ ]:
%%stata
gen x2=x^2
reg y x x2

我们计算拟合抛物线的对称轴位置 -β1/(2β2)，并存入局部宏，供图中画竖线使用，用来观察样本数据落在抛物线的哪一段。

In [ ]:
%%stata
local sym = _b[x]/(-2*_b[x2])

我们对 x 排序后画图：散点是模拟数据，实线是二次拟合曲线（带对称轴竖线），另一条是真实的条件期望曲线。图形直观展示了二次近似的适用范围与局限。

In [ ]:
%%stata
predict y_hat
sort x
twoway (scatter y x) (line y_hat x, xline(`sym')) (line y_true x)